In [2]:
# ======================================================================
# 01_build_corpus_graph_cases.ipynb
# FinGap-Governance | Materialize the research-authored artifacts.
# ----------------------------------------------------------------------
# Writes the three JSON artifacts the experiments consume:
#   data/corpus/<task>.json  -- RAG corpus (governing provisions + distractors)
#   data/graph/<task>.json   -- manually-curated provision-judgment graph
#   data/cases/<task>.json   -- balanced case bank (clear/boundary/distractor)
# All content is research-authored and PII-free. Then runs integrity checks
# (every gold provision is canonical) and prints the difficulty balance.
# ======================================================================
from fingap_core import *
import collections, json
import _build_artifacts as B

counts = B.write_all()
print("written under data/: corpus/, graph/, cases/")
print("case counts per task:", counts)

for task in TASKS:
    graph_ids = {n["id"] for n in load_graph(task).get("provision_nodes", [])}
    cases = load_cases(task)
    for c in cases:                                   # integrity: gold ids are canonical
        for p in c.gold_provisions:
            assert p in PROVISIONS, f"{c.case_id}: non-canonical gold {p}"
    bal = collections.Counter(c.difficulty for c in cases)
    prov = collections.Counter(c.provenance for c in cases)
    n_exp = sum(1 for c in cases if c.expert_adjudicated)
    print(f"  {task:11} n={len(cases):2d} difficulty={dict(bal)} "
          f"provenance={dict(prov)} expert_adjudicated={n_exp}/{len(cases)}")

print("\nNOTE: expert_adjudicated=False by design. A domain expert sets this True "
      "(and fills 'adjudicator') per case before the runs count as validated evidence.")


written under data/: corpus/, graph/, cases/
case counts per task: {'advisory': 18, 'anomaly': 18, 'accounting': 18}
  advisory    n=18 difficulty={'clear': 12, 'boundary': 4, 'distractor': 2} provenance={'research_authored': 18} expert_adjudicated=0/18
  anomaly     n=18 difficulty={'clear': 11, 'boundary': 5, 'distractor': 2} provenance={'research_authored': 18} expert_adjudicated=0/18
  accounting  n=18 difficulty={'clear': 12, 'boundary': 4, 'distractor': 2} provenance={'research_authored': 18} expert_adjudicated=0/18

NOTE: expert_adjudicated=False by design. A domain expert sets this True (and fills 'adjudicator') per case before the runs count as validated evidence.
